# Gold - Power BI Ready Tables

This notebook reads the cleaned `silver` tables and builds the `gold` tables.

Gold tables use a simple **star schema**:

- **Dimension tables** (`dim_`) describe things: customers, drivers, vehicles, warehouses, routes and dates.
- **Fact tables** (`fact_`) record what happened: shipments and delivery events.

In Power BI, each fact table connects to the dimension tables through the ID columns.

Run the cells from top to bottom.

In [ ]:
import os

import pandas as pd
from sqlalchemy import create_engine

DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Used to work out the age of a vehicle.
today = pd.Timestamp.today().normalize()

## Setup

In [ ]:
def get_silver(table_name):
    '''Read one table from the silver schema.'''
    return pd.read_sql(f"select * from silver.{table_name}", engine)


def load_to_gold(df, table_name):
    '''Replace the gold table with the DataFrame.'''
    df.to_sql(
        table_name,
        engine,
        schema="gold",
        if_exists="replace",
        index=False,
    )
    print(f"Loaded gold.{table_name}: {len(df):,} rows")

### Helper functions

These small functions are reused for every table.

In [ ]:
def keep_valid_keys(df, key_column):
    '''Drop rows with no ID and keep one row per ID.'''
    before = len(df)
    df = df[df[key_column].notna()]
    df = df.drop_duplicates(subset=key_column, keep="first")
    print(f"{key_column}: removed {before - len(df)} rows (missing or duplicate ID)")
    return df.reset_index(drop=True)


def fill_unknown(df, columns):
    '''Replace missing text with "Unknown", so Power BI shows a label and not a blank.'''
    for column in columns:
        df[column] = df[column].astype("string").fillna("Unknown")
    return df


def to_date(column):
    '''Keep only the date part and drop the time.'''
    return pd.to_datetime(column).dt.date

## Dimension tables

### dim_customers

In [ ]:
dim_customers = get_silver("customers")
dim_customers = keep_valid_keys(dim_customers, "customer_id")

# Email and phone are personal details. Reports do not need them.
dim_customers = dim_customers[
    [
        "customer_id",
        "customer_name",
        "gender",
        "customer_type",
        "city",
        "state",
        "pincode",
        "signup_date",
    ]
].copy()

dim_customers = fill_unknown(
    dim_customers,
    ["customer_name", "gender", "customer_type", "city", "state", "pincode"],
)
dim_customers["signup_date"] = to_date(dim_customers["signup_date"])

print("Rows:", len(dim_customers))
dim_customers.head()

### dim_drivers

In [ ]:
dim_drivers = get_silver("drivers")
dim_drivers = keep_valid_keys(dim_drivers, "driver_id")

# Phone and licence number are personal details. Reports do not need them.
dim_drivers = dim_drivers[
    [
        "driver_id",
        "driver_name",
        "gender",
        "city",
        "experience_years",
        "driver_rating",
        "join_date",
    ]
].copy()

dim_drivers = fill_unknown(dim_drivers, ["driver_name", "gender", "city"])
dim_drivers["experience_years"] = dim_drivers["experience_years"].astype("Int64")
dim_drivers["join_date"] = to_date(dim_drivers["join_date"])

print("Rows:", len(dim_drivers))
dim_drivers.head()

### dim_vehicles

In [ ]:
dim_vehicles = get_silver("vehicles")
dim_vehicles = keep_valid_keys(dim_vehicles, "vehicle_id")

dim_vehicles["year"] = dim_vehicles["year"].astype("Int64")

# Age of the vehicle in years.
dim_vehicles["vehicle_age"] = today.year - dim_vehicles["year"]

dim_vehicles = fill_unknown(
    dim_vehicles, ["vehicle_number", "vehicle_type", "make", "status"]
)

print("Rows:", len(dim_vehicles))
dim_vehicles.head()

### dim_warehouses

In [ ]:
dim_warehouses = get_silver("warehouses")
dim_warehouses = keep_valid_keys(dim_warehouses, "warehouse_id")

dim_warehouses["capacity_units"] = dim_warehouses["capacity_units"].astype("Int64")
dim_warehouses["operational_since"] = to_date(dim_warehouses["operational_since"])

dim_warehouses = fill_unknown(
    dim_warehouses,
    ["warehouse_name", "city", "state", "pincode", "manager_name"],
)

print("Rows:", len(dim_warehouses))
dim_warehouses.head()

### dim_routes

In [ ]:
dim_routes = get_silver("routes")
dim_routes = keep_valid_keys(dim_routes, "route_id")

dim_routes = fill_unknown(
    dim_routes,
    ["origin_city", "origin_state", "destination_city", "destination_state"],
)

# A readable label for charts, for example "Mumbai to Delhi".
dim_routes["route_name"] = (
    dim_routes["origin_city"] + " to " + dim_routes["destination_city"]
)

print("Rows:", len(dim_routes))
dim_routes.head()

## Fact tables

### fact_shipments

One row per shipment. This table adds delivery time, delay and cost columns,
so Power BI can build KPIs easily.

In [ ]:
fact_shipments = get_silver("shipments")
fact_shipments = keep_valid_keys(fact_shipments, "shipment_id")

# Make sure the date columns are real dates, so we can subtract them.
for column in ["shipment_date", "expected_delivery_date", "actual_delivery_date"]:
    fact_shipments[column] = pd.to_datetime(fact_shipments[column])

print("Rows:", len(fact_shipments))

In [ ]:
# Days are worked out while the columns are still dates with a time part.
shipment_date = fact_shipments["shipment_date"]
expected_date = fact_shipments["expected_delivery_date"]
actual_date = fact_shipments["actual_delivery_date"]

# Days promised to the customer.
fact_shipments["planned_days"] = (expected_date - shipment_date).dt.days.astype("Int64")

# Days the delivery really took.
fact_shipments["actual_days"] = (actual_date - shipment_date).dt.days.astype("Int64")

# Positive = late, zero or negative = on time or early.
fact_shipments["delay_days"] = (actual_date - expected_date).dt.days.astype("Int64")

print(fact_shipments[["planned_days", "actual_days", "delay_days"]].describe())

In [ ]:
# Unknown means the actual or expected date is missing, so we cannot judge.
fact_shipments["delivery_result"] = "Unknown"
fact_shipments.loc[fact_shipments["delay_days"] <= 0, "delivery_result"] = "On Time"
fact_shipments.loc[fact_shipments["delay_days"] > 0, "delivery_result"] = "Late"

# 1 or 0 columns make DAX measures simple (SUM instead of COUNTROWS + FILTER).
fact_shipments["is_on_time"] = (fact_shipments["delivery_result"] == "On Time").astype(int)
fact_shipments["is_late"] = (fact_shipments["delivery_result"] == "Late").astype(int)

print(fact_shipments["delivery_result"].value_counts())

In [ ]:
# Cost per km and cost per kg. Silver already removed zero and negative values.
fact_shipments["cost_per_km"] = (
    fact_shipments["shipping_cost_inr"] / fact_shipments["distance_km"]
).round(2)

fact_shipments["cost_per_kg"] = (
    fact_shipments["shipping_cost_inr"] / fact_shipments["weight_kg"]
).round(2)

fact_shipments["delivery_rating"] = fact_shipments["delivery_rating"].astype("Int64")

fact_shipments = fill_unknown(
    fact_shipments, ["package_type", "payment_method", "shipment_status"]
)

# Keep only the date part, so the dates match dim_date.
for column in ["shipment_date", "expected_delivery_date", "actual_delivery_date"]:
    fact_shipments[column] = to_date(fact_shipments[column])

In [ ]:
# Final column order: IDs first, then dates, then details and measures.
fact_shipments = fact_shipments[
    [
        "shipment_id",
        "customer_id",
        "warehouse_id",
        "driver_id",
        "vehicle_id",
        "route_id",
        "shipment_date",
        "expected_delivery_date",
        "actual_delivery_date",
        "package_type",
        "payment_method",
        "shipment_status",
        "weight_kg",
        "distance_km",
        "shipping_cost_inr",
        "cost_per_km",
        "cost_per_kg",
        "delivery_rating",
        "planned_days",
        "actual_days",
        "delay_days",
        "delivery_result",
        "is_on_time",
        "is_late",
    ]
]

fact_shipments.head()

### fact_delivery_events

One row per tracking event (picked up, in transit, delivered, and so on).

In [ ]:
fact_delivery_events = get_silver("delivery_events")
fact_delivery_events = keep_valid_keys(fact_delivery_events, "event_id")

# An event with no shipment cannot be linked to anything, so remove it.
before = len(fact_delivery_events)
fact_delivery_events = fact_delivery_events[
    fact_delivery_events["shipment_id"].notna()
].reset_index(drop=True)
print("Events removed (no shipment ID):", before - len(fact_delivery_events))

# Date-only column, used to link the event to dim_date.
fact_delivery_events["event_date"] = to_date(fact_delivery_events["event_timestamp"])

fact_delivery_events = fill_unknown(
    fact_delivery_events, ["event_type", "location_city", "remarks"]
)

print("Rows:", len(fact_delivery_events))
fact_delivery_events.head()

## dim_date

A calendar table. It has one row for every day, from 1 January of the first
year in the data to 31 December of the last year.

In [ ]:
all_dates = pd.concat(
    [
        fact_shipments["shipment_date"],
        fact_shipments["expected_delivery_date"],
        fact_shipments["actual_delivery_date"],
        fact_delivery_events["event_date"],
    ]
)
all_dates = pd.to_datetime(all_dates).dropna()

start = pd.Timestamp(year=all_dates.min().year, month=1, day=1)
end = pd.Timestamp(year=all_dates.max().year, month=12, day=31)

dim_date = pd.DataFrame({"full_date": pd.date_range(start, end, freq="D")})
day = dim_date["full_date"]

dim_date["year"] = day.dt.year
dim_date["quarter"] = "Q" + day.dt.quarter.astype("string")
dim_date["month_number"] = day.dt.month
dim_date["month_name"] = day.dt.month_name()
dim_date["year_month"] = day.dt.strftime("%Y-%m")
dim_date["day"] = day.dt.day
dim_date["weekday_number"] = day.dt.dayofweek + 1  # 1 = Monday
dim_date["weekday_name"] = day.dt.day_name()

dim_date["day_type"] = "Weekday"
dim_date.loc[day.dt.dayofweek >= 5, "day_type"] = "Weekend"

dim_date["full_date"] = day.dt.date

print("Date range:", dim_date["full_date"].min(), "to", dim_date["full_date"].max())
dim_date.head()

## Load

Write all gold tables to the `gold` schema.

In [ ]:
load_to_gold(dim_customers, "dim_customers")
load_to_gold(dim_drivers, "dim_drivers")
load_to_gold(dim_vehicles, "dim_vehicles")
load_to_gold(dim_warehouses, "dim_warehouses")
load_to_gold(dim_routes, "dim_routes")
load_to_gold(dim_date, "dim_date")
load_to_gold(fact_shipments, "fact_shipments")
load_to_gold(fact_delivery_events, "fact_delivery_events")

## Validation

Check that every table has a unique ID, and that every link between tables
points to a row that exists.

In [ ]:
tables = {
    "dim_customers": "customer_id",
    "dim_drivers": "driver_id",
    "dim_vehicles": "vehicle_id",
    "dim_warehouses": "warehouse_id",
    "dim_routes": "route_id",
    "dim_date": "full_date",
    "fact_shipments": "shipment_id",
    "fact_delivery_events": "event_id",
}

print("GOLD VALIDATION")
print("-" * 60)

for table, key in tables.items():
    df = pd.read_sql(f"select * from gold.{table}", engine)
    print(
        f"{table}: rows={len(df)}, "
        f"null IDs={df[key].isna().sum()}, "
        f"duplicate IDs={df[key].dropna().duplicated().sum()}"
    )

In [ ]:
# (table with the link, link column, table it points to, ID column there)
links = [
    ("fact_shipments", "customer_id", "dim_customers", "customer_id"),
    ("fact_shipments", "driver_id", "dim_drivers", "driver_id"),
    ("fact_shipments", "vehicle_id", "dim_vehicles", "vehicle_id"),
    ("fact_shipments", "warehouse_id", "dim_warehouses", "warehouse_id"),
    ("fact_shipments", "route_id", "dim_routes", "route_id"),
    ("fact_shipments", "shipment_date", "dim_date", "full_date"),
    ("fact_delivery_events", "shipment_id", "fact_shipments", "shipment_id"),
    ("fact_delivery_events", "event_date", "dim_date", "full_date"),
]

print("LINK CHECK (links that point to nothing)")
print("-" * 60)

for from_table, from_column, to_table, to_column in links:
    from_values = pd.read_sql(
        f"select {from_column} from gold.{from_table}", engine
    )[from_column]
    to_values = pd.read_sql(
        f"select {to_column} from gold.{to_table}", engine
    )[to_column]

    broken = from_values.notna() & ~from_values.isin(to_values)
    missing = from_values.isna().sum()
    print(
        f"{from_table}.{from_column} -> {to_table}.{to_column}: "
        f"broken={broken.sum()}, empty={missing}"
    )

In [ ]:
# Quick KPI check. Compare these numbers with your Power BI cards later.
shipments = pd.read_sql("select * from gold.fact_shipments", engine)
judged = shipments[shipments["delivery_result"] != "Unknown"]

print("Total shipments:", len(shipments))
print("On-time delivery %:", round(judged["is_on_time"].mean() * 100, 1))
print("Average delay (days):", round(judged["delay_days"].mean(), 2))
print("Total shipping cost (INR):", round(shipments["shipping_cost_inr"].sum(), 2))
print("Average delivery rating:", round(shipments["delivery_rating"].mean(), 2))